# Explorar y caracterizar un clustering de DTW

`02_DTW_clusters` **genera** los clusters. Este notebook los **abre**: toma un
`clusters_dtw_*.parquet` ya guardado y contesta cuatro preguntas, en orden:

1. ¿Cuántas entidades quedaron en cada cluster, y cuánto volumen representan?
2. ¿Quiénes son? ¿Me quedaron las sopas juntas, o es una bolsa de gatos?
3. ¿Cómo se ven las series **en la escala normalizada**, que es la que vio DTW?
4. ¿Y **en toneladas**? Porque ahí se descubre qué unió de verdad.

La 3 y la 4 juntas son el punto. DTW agrupó dos productos porque *normalizados* se
parecen; mirándolos en toneladas descubrís si eso significa algo — que uno vende 1000
y el otro 5 con la misma forma es un hallazgo, no un error.

Cada salida viene con un bloque **«Qué mirar»** que dice qué es señal buena y qué es
motivo para descartar la configuración.

> **Sobre Plotly**: los gráficos se guardan **siempre** como HTML en la carpeta del
> clustering, además de mostrarse. Si en tu JupyterLab `fig.show()` no renderiza (falta
> la extensión), abrí el `.html` — está todo ahí, interactivo.

## 0 — Ambiente

In [ ]:
import os, json
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    local = Path(r"C:\Users\Natalia\labo3-bucket")
    if local.is_dir():
        return local
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_FE   = BUCKET / "datasets_fe"
DIR_RUNS = BUCKET / "exp_clusters"

# Paleta: una serie por color, y un gris para el resto.
COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
           "#008300", "#4a3aa7", "#e34948", "#0f8b8d", "#b5651d",
           "#7a5195", "#ef5675"]
GRIS, TINTA = "#b8b8b0", "#0b0b0b"

PLANTILLA = dict(
    template="simple_white",
    font=dict(size=12, color=TINTA),
    margin=dict(l=60, r=20, t=60, b=50),
    hovermode="closest",
)

print(f"BUCKET: {BUCKET}")
print(f"\nClusterings disponibles en {DIR_FE.name}:")
disponibles = sorted(DIR_FE.glob("clusters_dtw_*.parquet"))
for p in disponibles:
    print(f"  - {p.name}")
if not disponibles:
    raise RuntimeError(f"No hay clusters_dtw_*.parquet en {DIR_FE}. Corre 02_DTW_clusters.")

## 1 — Qué clustering se explora

El único parámetro obligatorio es el archivo. Todo lo demás — nivel, eje,
normalización, ventana, corte — **viene guardado adentro del parquet**, así que la
reconstrucción de las series usa exactamente la misma configuración con la que se
clusterizó. No hay forma de mirar un clustering con la normalización equivocada.

In [ ]:
PARAM = {
    # Nombre exacto de la lista de arriba. None = el ultimo por orden alfabetico.
    'archivo': None,

    # Que cluster se explora en detalle (las secciones 3 y 4). None = el mas grande.
    'cluster': None,

    # Cuantas series se plotean del cluster elegido. Se toman las N de mayor volumen
    # mas N al azar: las grandes son las que mueven el WAPE, las al azar evitan que te
    # quedes con la impresion de las grandes nada mas.
    'n_top': 6,
    'n_azar': 6,

    'semilla': 102191,
}

nombre = PARAM['archivo'] or disponibles[-1].name
path_clu = DIR_FE / nombre
if not path_clu.exists():
    raise FileNotFoundError(f"No existe {path_clu}")

clu = pl.read_parquet(path_clu)

# ── La config viaja dentro del parquet ───────────────────────────────────
CFG = {c.replace("dtw_", ""): clu[c][0] for c in clu.columns if c.startswith("dtw_")}

# Los parquet generados ANTES de que 02_DTW tuviera la palanca 'entidad' no guardan
# entidad, norm, window ni linkage, y su columna de cluster se llama 'cluster_dtw' a
# secas. Sin la normalizacion no se pueden reconstruir las series que vio DTW, asi que
# no hay forma de explorarlos: hay que regenerarlos.
_faltan = [k for k in ("entidad", "eje", "norm", "mes_corte") if k not in CFG]
if _faltan:
    raise RuntimeError(
        f"{nombre} es de formato viejo: le faltan las columnas {_faltan}. "
        f"Sin la normalizacion no se pueden reconstruir las series que vio DTW. "
        f"Regeneralo corriendo 02_DTW_clusters, que ya deja el nombre nuevo con todas "
        f"las palancas, y volve a esta celda."
    )

ENTIDAD = CFG['entidad']
KEY = 'product_id' if ENTIDAD == 'producto' else 'customer_id'
COL_CLUSTER = f"cluster_dtw_{ENTIDAD}"
ES_PRODUCTO = ENTIDAD == 'producto'

SLUG = nombre.replace("clusters_dtw_", "").replace(".parquet", "")
DIR_RUN = DIR_RUNS / SLUG.rsplit("_k", 1)[0]
DIR_RUN.mkdir(parents=True, exist_ok=True)

print(f"archivo : {nombre}")
print(f"entidad : {ENTIDAD}   (clave: {KEY})")
print(f"config  : {CFG}")
print(f"graficos: {DIR_RUN.relative_to(BUCKET)}")
print(f"\n{clu.height} {ENTIDAD}s etiquetados en {clu[COL_CLUSTER].n_unique()} clusters")

## 2 — Reconstruir las series como las vio DTW

Mismo panel, mismo corte y misma normalización que usó `02_DTW_clusters`, leídos de la
config del parquet. Al final hay un chequeo: las entidades reconstruidas tienen que ser
**exactamente** las mismas que están etiquetadas en el archivo. Si no coinciden, alguna
de las dos implementaciones se movió y los gráficos mentirían.

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))


def a_m(col):
    return (pl.col(col) // 100) * 12 + (pl.col(col) % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


panel = (sell.group_by([KEY, "periodo"]).agg(pl.col("tn").sum().alias("tn"))
             .with_columns(a_m("periodo").alias("m")).sort([KEY, "m"]))

if CFG['mes_corte'] is not None:
    m_corte = (int(CFG['mes_corte']) // 100) * 12 + (int(CFG['mes_corte']) % 100)
    panel = panel.filter(pl.col("m") < m_corte)

vida = panel.group_by(KEY).agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"),
    pl.col("tn").sum().alias("tn_total"))

grilla = (vida.select(KEY, "m_nace", "m_muere")
              .with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").drop("m_nace", "m_muere"))

panel = (grilla.join(panel.drop("periodo"), on=[KEY, "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .join(vida, on=KEY, how="left")
               .with_columns((pl.col("m") - pl.col("m_nace")).alias("edad"))
               .sort([KEY, "m"]))


def normalizar(v: np.ndarray) -> np.ndarray:
    """Misma funcion que 02_DTW_clusters. Si la tocas alla, tocala aca."""
    n = CFG['norm']
    if n == "zscore":
        sd = v.std()
        return (v - v.mean()) / (sd if sd > 0 else 1.0)
    if n == "minmax":
        rango = v.max() - v.min()
        return (v - v.min()) / (rango if rango > 0 else 1.0)
    if n == "recta":
        t = np.arange(len(v), dtype=np.double)
        b1, b0 = np.polyfit(t, v, 1)
        r = v - (b0 + b1 * t)
        sd = r.std()
        return r / (sd if sd > 0 else 1.0)
    mx = v.max()
    return v / (mx if mx > 0 else 1.0)


IDS = clu[KEY].to_list()
GRILLA_M = list(range(int(panel["m"].min()), int(panel["m"].max()) + 1))
_pos = {m: i for i, m in enumerate(GRILLA_M)}

sub = panel.filter(pl.col(KEY).is_in(IDS)).select(KEY, "m", "tn").sort([KEY, "m"])

# SERIE_NORM: lo que vio DTW.  SERIE_TN: las toneladas originales, mismo eje.
SERIE_NORM, SERIE_TN, MESES_DE = {}, {}, {}
for pid, g in sub.group_by(KEY, maintain_order=True):
    pid = pid[0] if isinstance(pid, tuple) else pid
    ms = g["m"].to_list()
    tn = g["tn"].to_numpy().astype(np.double)
    if CFG['eje'] == "calendario":
        v = np.zeros(len(GRILLA_M)); v[[_pos[m] for m in ms]] = tn
        t = np.zeros(len(GRILLA_M)); t[[_pos[m] for m in ms]] = tn
        eje = [m_a_periodo(m) for m in GRILLA_M]
    else:
        v, t = tn.copy(), tn.copy()
        eje = list(range(len(tn)))
    SERIE_NORM[pid] = normalizar(v)
    SERIE_TN[pid] = t
    MESES_DE[pid] = eje

faltan = set(IDS) - set(SERIE_NORM)
if faltan:
    raise RuntimeError(f"{len(faltan)} {ENTIDAD}s etiquetados que no pude reconstruir: "
                       f"{sorted(faltan)[:5]}. La logica del panel se desincronizo de 02.")
print(f"{len(SERIE_NORM)} series reconstruidas, todas las etiquetadas. OK")
print(f"eje: {CFG['eje']}   normalizacion: {CFG['norm']}   largo: "
      f"{len(next(iter(SERIE_NORM.values())))} puntos")

## 3 — Cuántos quedaron en cada cluster

### Qué mirar

| Señal | Qué significa |
|---|---|
| **Un cluster con menos del 5 %** | Inservible: no alcanza ni como feature categórica ni para un modelo propio. Bajá `k` o cambiá de linkage. |
| **Un cluster con más del 80 %** | El clustering no separó nada: desprendió unos outliers y dejó todo lo demás junto. Es el síntoma típico de linkage `average` o `complete`. |
| **`%_tn` muy distinto de `%_entidades`** | Un cluster junta las grandes y otro las chicas → probablemente agrupó por **volumen**, no por forma. Con la normalización correcta esto no debería pasar; si pasa, revisá la normalización. |
| Reparto parejo en ambas columnas | Buena señal: separó por forma y no por tamaño. |

In [ ]:
resumen = (clu.join(vida.select(KEY, "tn_total"), on=KEY, how="left", suffix="_v")
              .group_by(COL_CLUSTER)
              .agg(pl.len().alias("n"),
                   pl.col("tn_total").sum().alias("tn"),
                   pl.col("tn_total").median().alias("tn_mediana"))
              .with_columns(
                  (100 * pl.col("n") / pl.col("n").sum()).round(1).alias("%_entidades"),
                  (100 * pl.col("tn") / pl.col("tn").sum()).round(1).alias("%_tn"))
              .sort(COL_CLUSTER))
print(resumen)

_n, _pe, _pt = resumen["n"], resumen["%_entidades"], resumen["%_tn"]
print()
if _pe.min() < 5:
    print(f"  AVISO: el cluster mas chico tiene {_pe.min()}% de las entidades (<5%): inservible.")
if _pe.max() > 80:
    print(f"  AVISO: un cluster concentra {_pe.max()}% de las entidades: no separo nada.")
_desb = float((resumen["%_tn"] - resumen["%_entidades"]).abs().max())
print(f"  desbalance maximo |%_tn - %_entidades| = {_desb:.1f} puntos"
      + ("   <- alto: puede estar agrupando por volumen" if _desb > 25 else "   <- razonable"))

CL = int(PARAM['cluster'] if PARAM['cluster'] is not None
         else resumen.sort("n", descending=True)[COL_CLUSTER][0])
miembros = clu.filter(pl.col(COL_CLUSTER) == CL)[KEY].to_list()
print(f"\nCluster que se explora: {CL}   ({len(miembros)} {ENTIDAD}s)")

## 4 — Quiénes son: ¿me quedaron las sopas juntas?

Dos tablas. La primera son los miembros con su descripción, para leer con los ojos si
tienen algo en común. La segunda es la que evita autoengañarse: mide
**sobrerrepresentación**, o sea si una categoría aparece más en este cluster que en el
conjunto.

### Qué mirar

| Señal | Qué significa |
|---|---|
| Una `cat3` con **lift > 2** y varios miembros | El cluster capturó algo comercialmente real (las sopas, las gaseosas). |
| Todos los lift **cerca de 1** | El cluster **no** coincide con la jerarquía. **No es un problema — es el objetivo**: agrupa por comportamiento temporal, que es información que `cat3` no tiene. |
| Una sola `cat3` con lift enorme y el resto vacío | Sospechoso: puede ser un cluster de un solo producto y sus variantes de tamaño, no un patrón. |

El caso «todos cerca de 1» es el bueno para el pipe. Si el cluster fuera igual a
`cat3`, el modelo ya tiene `cat3` y la feature nueva no aportaría nada.

In [ ]:
if ES_PRODUCTO:
    det = (clu.filter(pl.col(COL_CLUSTER) == CL)
              .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand",
                                "sku_size", "descripcion"),
                    on="product_id", how="left")
              .join(vida.select(KEY, "tn_total"), on=KEY, how="left", suffix="_v")
              .sort("tn_total", descending=True))
    cols_ver = ["product_id", "descripcion", "cat3", "brand", "sku_size", "tn_total"]
    print(f"=== Los 15 de mayor volumen del cluster {CL} ===")
    with pl.Config(tbl_rows=15, fmt_str_lengths=42, tbl_width_chars=160):
        print(det.select([c for c in cols_ver if c in det.columns]).head(15))

    # ── Sobrerrepresentacion: lift = share en el cluster / share en el total ──
    base = (clu.join(prod.select("product_id", "cat3"), on="product_id", how="left")
               .group_by("cat3").agg(pl.len().alias("n_total")))
    en_cl = det.group_by("cat3").agg(pl.len().alias("n_cluster"))
    lift = (en_cl.join(base, on="cat3", how="left")
                 .with_columns(
                     (100 * pl.col("n_cluster") / len(miembros)).round(1).alias("%_en_cluster"),
                     (100 * pl.col("n_total") / clu.height).round(1).alias("%_en_total"))
                 .with_columns((pl.col("%_en_cluster") / pl.col("%_en_total")).round(2).alias("lift"))
                 .sort("lift", descending=True))
    print(f"\n=== Categorias sobrerrepresentadas (lift > 1 = mas de lo esperado) ===")
    with pl.Config(tbl_rows=12, fmt_str_lengths=34):
        print(lift.filter(pl.col("n_cluster") >= 2).head(12))
    lift.write_csv(DIR_RUN / f"lift_cat3_cluster{CL}.csv")

    _lmax = float(lift.filter(pl.col("n_cluster") >= 3)["lift"].max() or 0)
    print(f"\nlift maximo (con >=3 miembros): {_lmax:.2f}")
    if _lmax < 1.5:
        print("  -> el cluster NO coincide con cat3: agrupa por comportamiento.")
        print("     Es la buena noticia: aporta informacion que la jerarquia no tiene.")
    else:
        print("  -> hay categorias concentradas: el cluster reproduce parte de la jerarquia.")
else:
    det = (clu.filter(pl.col(COL_CLUSTER) == CL)
              .join(vida.select(KEY, "tn_total"), on=KEY, how="left", suffix="_v")
              .sort("tn_total", descending=True))
    print(f"=== Los 15 clientes de mayor volumen del cluster {CL} ===")
    with pl.Config(tbl_rows=15):
        print(det.select("customer_id", "tn_total").head(15))
    print("\nLos clientes no tienen jerarquia comercial, asi que no hay lift por categoria.")
    print("El contraste equivalente es el volumen: mira si el cluster junta a los grandes.")

## 5 — Las series como las vio DTW (escala normalizada)

Esto es literalmente lo que se le pasó al algoritmo. Se plotean las de mayor volumen
más algunas al azar.

### Qué mirar

| Señal | Qué significa |
|---|---|
| Las curvas **se superponen bastante** | La normalización funcionó y DTW agrupó por forma. Es lo que buscás. |
| Curvas con la **misma forma pero corridas** en el eje | Perfecto: es justamente lo que DTW sabe alinear y la euclídea no. Confirma que valía usar DTW. |
| Una curva **muy distinta** de las demás | Un outlier metido a la fuerza. Si hay varios, subí `k` para que se separen. |
| Curvas **planas** salvo un pico | Ojo con la normalización: si es `pico`, un mes extraordinario aplasta todo el resto de la serie. Probá `zscore` o `minmax`. |
| Todas planas en 0 con saltos aislados | Series demasiado ralas para que DTW diga algo. Subí `min_meses`. |

In [ ]:
rng = np.random.default_rng(PARAM['semilla'])
_ord = det[KEY].to_list()                                   # ya viene ordenado por volumen
top = _ord[:PARAM['n_top']]
resto = [p for p in _ord[PARAM['n_top']:]]
azar = list(rng.choice(resto, size=min(PARAM['n_azar'], len(resto)), replace=False)) if resto else []
MUESTRA = top + [int(p) for p in azar]
print(f"{len(MUESTRA)} series: {len(top)} de mayor volumen + {len(azar)} al azar")

tn_de = dict(zip(det[KEY].to_list(), det["tn_total"].to_list()))
etiqueta = {}
for p in MUESTRA:
    if ES_PRODUCTO:
        d = det.filter(pl.col("product_id") == p)
        desc = (d["descripcion"][0] if "descripcion" in d.columns and d.height else "") or ""
        etiqueta[p] = f"{p} · {str(desc)[:26]} · {tn_de.get(p, 0):,.0f} tn"
    else:
        etiqueta[p] = f"cliente {p} · {tn_de.get(p, 0):,.0f} tn"

x_lbl = "período" if CFG['eje'] == "calendario" else "meses desde el primer mes"

fig = go.Figure()
for i, p in enumerate(MUESTRA):
    fig.add_trace(go.Scatter(
        x=MESES_DE[p], y=SERIE_NORM[p], mode="lines",
        name=etiqueta[p], line=dict(width=2, color=COLORES[i % len(COLORES)]),
        hovertemplate="%{fullData.name}<br>%{x}: %{y:.2f}<extra></extra>"))
fig.update_layout(
    title=f"Cluster {CL} — escala NORMALIZADA ({CFG['norm']}), lo que vio DTW",
    xaxis_title=x_lbl, yaxis_title=f"tn {CFG['norm']}",
    legend=dict(font=dict(size=9)), height=460, **PLANTILLA)
_p = DIR_RUN / f"cluster{CL}_normalizada.html"
fig.write_html(_p)
print(f"guardado: {_p.relative_to(BUCKET)}")
fig.show()

## 6 — Las mismas series en toneladas: qué unió de verdad

**Éste es el gráfico que importa.** El de arriba muestra por qué DTW las juntó; éste
muestra qué significa. Son las mismas entidades, mismo orden, mismos colores.

### Qué mirar

| Señal | Qué significa |
|---|---|
| Curvas **muy separadas en altura** pero paralelas | **El resultado ideal.** La normalización hizo su trabajo: unió series de escalas distintas con la misma forma. Es información que ninguna feature de nivel te da. |
| Curvas **todas a la misma altura** | Sospechoso: puede que en realidad haya agrupado por volumen y la normalización no esté haciendo nada. Cruzá con el desbalance `%_tn` vs `%_entidades` de la sección 3. |
| Una domina y el resto es plano abajo | El cluster está dominado por un producto grande. Como feature sirve poco: el modelo ya distingue a ese producto solo. |
| Escala log necesaria para ver algo | Normal con toneladas. El botón de arriba a la derecha cambia entre lineal y log. |

El primer caso es el que justifica todo el ejercicio. Si lo ves, tenés la respuesta
para el profe: DTW + normalización encontró familias de comportamiento que atraviesan
escalas, y eso la distancia euclídea sin normalizar no lo puede hacer.

In [ ]:
fig = go.Figure()
for i, p in enumerate(MUESTRA):
    fig.add_trace(go.Scatter(
        x=MESES_DE[p], y=SERIE_TN[p], mode="lines",
        name=etiqueta[p], line=dict(width=2, color=COLORES[i % len(COLORES)]),
        hovertemplate="%{fullData.name}<br>%{x}: %{y:,.1f} tn<extra></extra>"))
fig.update_layout(
    title=f"Cluster {CL} — TONELADAS originales (mismas series, mismos colores)",
    xaxis_title=x_lbl, yaxis_title="toneladas",
    legend=dict(font=dict(size=9)), height=460,
    updatemenus=[dict(
        type="buttons", direction="right", x=1, xanchor="right", y=1.13,
        buttons=[dict(label="lineal", method="relayout", args=[{"yaxis.type": "linear"}]),
                 dict(label="log", method="relayout", args=[{"yaxis.type": "log"}])])],
    **PLANTILLA)
_p = DIR_RUN / f"cluster{CL}_toneladas.html"
fig.write_html(_p)
print(f"guardado: {_p.relative_to(BUCKET)}")
fig.show()

# ── El numero detras del grafico ─────────────────────────────────────────
_tns = np.array([tn_de.get(p, 0.0) for p in MUESTRA], dtype=float)
_tns = _tns[_tns > 0]
if len(_tns) > 1:
    print(f"\nvolumen de las series ploteadas:  min {_tns.min():,.0f} tn   "
          f"max {_tns.max():,.0f} tn   ratio max/min = {_tns.max()/_tns.min():.0f}x")
    if _tns.max() / _tns.min() > 10:
        print("  -> unio series con mas de un orden de magnitud de diferencia.")
        print("     Eso es exactamente lo que la normalizacion viene a permitir.")
    else:
        print("  -> todas de escala parecida: puede estar agrupando por volumen.")

## 7 — El centroide y la dispersión del cluster

La mediana punto a punto de **todos** los miembros (no sólo la muestra), con la banda
entre el percentil 25 y el 75. Es el resumen de «la forma» del cluster.

### Qué mirar

| Señal | Qué significa |
|---|---|
| Banda **angosta** alrededor de la mediana | Cluster compacto: sus miembros de verdad se parecen. |
| Banda **tan ancha como el rango** de los datos | El cluster no tiene una forma propia; es un cajón de sastre. |
| Centroides de distintos clusters **parecidos entre sí** | `k` es demasiado alto: estás partiendo un grupo homogéneo. Bajalo. |
| Centroides claramente distintos | `k` razonable. Si además la banda es angosta, la configuración sirve. |

El último panel compara los centroides de **todos** los clusters, que es la vista que
de un golpe te dice si el clustering separó algo o no.

In [ ]:
K = int(clu[COL_CLUSTER].max())
LARGO = int(np.median([len(SERIE_NORM[p]) for p in IDS]))

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"Cluster {CL}: mediana y banda 25–75 (n={len(miembros)})",
    f"Los {K} centroides, superpuestos"))

# panel izquierdo: el cluster elegido, con su dispersion
M = np.full((len(miembros), LARGO), np.nan)
for r, p in enumerate(miembros):
    s = SERIE_NORM[p]
    M[r, :min(len(s), LARGO)] = s[:LARGO]
med = np.nanmedian(M, axis=0)
q25 = np.nanpercentile(M, 25, axis=0)
q75 = np.nanpercentile(M, 75, axis=0)
t = np.arange(LARGO)

fig.add_trace(go.Scatter(x=np.concatenate([t, t[::-1]]),
                         y=np.concatenate([q75, q25[::-1]]),
                         fill="toself", fillcolor="rgba(42,120,214,.18)",
                         line=dict(width=0), name="p25–p75",
                         hoverinfo="skip"), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=med, mode="lines", name=f"mediana cluster {CL}",
                         line=dict(width=3, color=COLORES[0])), row=1, col=1)

# panel derecho: todos los centroides
for c in range(1, K + 1):
    mm = clu.filter(pl.col(COL_CLUSTER) == c)[KEY].to_list()
    if not mm:
        continue
    Mc = np.full((len(mm), LARGO), np.nan)
    for r, p in enumerate(mm):
        s = SERIE_NORM.get(p)
        if s is None:
            continue
        Mc[r, :min(len(s), LARGO)] = s[:LARGO]
    fig.add_trace(go.Scatter(x=t, y=np.nanmedian(Mc, axis=0), mode="lines",
                             name=f"cluster {c} (n={len(mm)})",
                             line=dict(width=3 if c == CL else 2,
                                       color=COLORES[(c - 1) % len(COLORES)])),
                  row=1, col=2)

fig.update_layout(height=420, legend=dict(font=dict(size=9)),
                  title_text=f"Formas típicas · {ENTIDAD}s · norm={CFG['norm']} · eje={CFG['eje']}",
                  **PLANTILLA)
fig.update_xaxes(title_text="posición en la serie")
fig.update_yaxes(title_text=f"tn {CFG['norm']}", row=1, col=1)
_p = DIR_RUN / f"centroides_k{K}.html"
fig.write_html(_p)
print(f"guardado: {_p.relative_to(BUCKET)}")
fig.show()

# ── Ancho de la banda: el numero que resume la dispersion ────────────────
_ancho = float(np.nanmean(q75 - q25))
_rango = float(np.nanmax(med) - np.nanmin(med))
print(f"\nancho medio de la banda p25-p75 : {_ancho:.3f}")
print(f"amplitud de la mediana           : {_rango:.3f}")
print(f"ratio ancho/amplitud             : {_ancho/_rango:.2f}" if _rango > 0 else "")
if _rango > 0 and _ancho / _rango > 1.5:
    print("  -> la dispersion supera a la senial: cluster poco compacto.")
else:
    print("  -> la forma del cluster se distingue de su propia dispersion. Bien.")

## 8 — Qué hacer con esto

Este notebook **no** decide nada: describe. El veredicto lo da el WAPE.

**Descartá la configuración** si viste alguna de estas:

- algún cluster con menos del 5 % de las entidades, o uno con más del 80 %
- desbalance `%_tn` vs `%_entidades` mayor a 25 puntos (agrupó por volumen)
- en toneladas, todas las series a la misma altura (la normalización no hizo nada)
- banda de dispersión más ancha que la señal del centroide
- centroides de distintos clusters casi iguales (bajá `k`)

**Llevala al pipe** si viste esto:

- reparto razonable de entidades y de volumen
- en normalizado, curvas que se superponen o que tienen la misma forma corrida
- en toneladas, **curvas separadas en altura pero paralelas** — es la prueba de que
  unió familias de comportamiento a través de escalas
- lift de `cat3` cerca de 1: el cluster aporta algo que la jerarquía no tiene

**Cómo llevarla**: en `02_FE`, un `join` por `product_id` (o `customer_id`) pegando la
columna del cluster, agregándola a `cols_categoricas` de `03_Optuna`, con
`sufijo='conCluster'` para que no pise el experimento base. Y comparás `wape_test` en
el leaderboard.

Para los que quedan sin etiqueta (menos de `min_meses` de historia) conviene una
categoría explícita `sin_cluster`: «poca historia» es en sí misma una señal.

In [ ]:
print("Archivos de esta exploracion:")
for p in sorted(DIR_RUN.iterdir()):
    if p.suffix in (".html", ".csv"):
        print(f"  - {p.name}")
print(f"\nen {DIR_RUN.relative_to(BUCKET)}")
print(f"\nPara explorar otro cluster: PARAM['cluster'] = <n> y volve a correr desde la seccion 3.")
print(f"Para otro clustering: PARAM['archivo'] = '<nombre>.parquet' y corre todo de nuevo.")